In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import os

2026-07-05 05:47:09.901130: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-05 05:47:10.483977: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-05 05:47:12.871903: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [5]:
# ====================== CONFIGURATION ======================
MODEL_PATH = "/mnt/c/Users/AbdulHafeez/Brain_Tumor/models/best_efficientnetv2s.keras"
IMG_SIZE = 224
CLASS_NAMES = ['glioma', 'meningioma', 'pituitary', 'no_tumor']

In [6]:
# ====================== LOAD MODEL ======================
print("Loading model...")
model = load_model(MODEL_PATH)
model.summary()

Loading model...


I0000 00:00:1783230607.470890     858 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1751 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Ti Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "EfficientNetV2S_BrainTumor"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        648 │ rescaling[0][0]   │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │         96 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │      5,184 │ stem_activation[… │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 112, 112,  │         96 │ block1a_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 112, 112,  │          0 │ block1a_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_add (Add)   │ (None, 112, 112,  │          0 │ block1a_project_… │
│                     │ 24)               │            │ stem_activation[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_co… │ (None, 112, 112,  │      5,184 │ block1a_add[0][0] │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_bn  │ (None, 112, 112,  │         96 │ block1b_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_ac… │ (None, 112, 112,  │          0 │ block1b_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_drop        │ (None, 112, 112,  │          0 │ block1b_project_… │
│ (Dropout)           │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_add (Add)   │ (None, 112, 112,  │          0 │ block1b_drop[0][… │
│                     │ 24)               │            │ block1a_add[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_conv │ (None, 56, 56,    │     20,736 │ block1b_add[0][0] │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_bn   │ (None, 56, 56,    │        384 │ block2a_expand_c… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_act… │ (None, 56, 56,    │          0 │ block2a_expand_b

 Total params: 41,588,718 (158.65 MB)

 Trainable params: 10,546,180 (40.23 MB)

 Non-trainable params: 9,950,176 (37.96 MB)

 Optimizer params: 21,092,362 (80.46 MB)

In [7]:
# ====================== GRAD-CAM FUNCTION ======================
def get_gradcam_heatmap(model, img_array, last_conv_layer_name, pred_index=None):
    """Generate Grad-CAM heatmap"""
    grad_model = tf.keras.models.Model(
        [model.inputs], 
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

In [8]:
# ====================== PREPROCESS IMAGE ======================
def preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = tf.keras.applications.efficientnet_v2.preprocess_input(img_array)
    return img_array, img

In [9]:
# ====================== MAIN FUNCTION ======================
def generate_gradcam(img_path, output_folder="gradcam_results"):
    os.makedirs(output_folder, exist_ok=True)
    
    img_array, original_img = preprocess_image(img_path)
    
    # Get prediction
    preds = model.predict(img_array)
    predicted_class = CLASS_NAMES[np.argmax(preds[0])]
    confidence = np.max(preds[0]) * 100
    
    print(f"Predicted: {predicted_class} ({confidence:.2f}%)")
    
    # Find the last convolutional layer (important!)
    last_conv_layer_name = None
    for layer in reversed(model.layers):
        if 'conv' in layer.name or 'fused' in layer.name or 'mbconv' in layer.name:
            last_conv_layer_name = layer.name
            break
    
    if not last_conv_layer_name:
        print("Could not find conv layer. Using last layer instead.")
        last_conv_layer_name = model.layers[-3].name  # fallback
    
    print(f"Using layer: {last_conv_layer_name}")
    
    # Generate heatmap
    heatmap = get_gradcam_heatmap(model, img_array, last_conv_layer_name)
    
    # Resize heatmap to original image size
    heatmap = cv2.resize(heatmap, (original_img.size[0], original_img.size[1]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    
    # Superimpose
    original_array = np.array(original_img)
    superimposed = heatmap * 0.4 + original_array * 0.6
    superimposed = np.uint8(superimposed)
    
    # Save result
    filename = os.path.basename(img_path)
    output_path = os.path.join(output_folder, f"gradcam_{filename}")
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 3, 1)
    plt.title("Original")
    plt.imshow(original_array)
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.title("Grad-CAM Heatmap")
    plt.imshow(heatmap)
    plt.axis('off')
    
    plt.subplot(1, 3, 3)
    plt.title(f"Prediction: {predicted_class} ({confidence:.1f}%)")
    plt.imshow(superimposed.astype(np.uint8))
    plt.axis('off')
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Grad-CAM saved: {output_path}\n")

In [11]:
# ====================== RUN ON MULTIPLE IMAGES ======================
if __name__ == "__main__":
    # Test folder path
    test_folder = "/mnt/c/Users/AbdulHafeez/Brain_Tumor/Data_Split/test"
    
    # Get all image files from test folder
    test_images = []
    for root, dirs, files in os.walk(test_folder):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                test_images.append(os.path.join(root, file))
    
    print(f"Found {len(test_images)} test images.")
    
    # Limit to first 10-15 images to avoid long time (you can increase)
    test_images = test_images[:15]  
    
    for img_path in test_images:
        if os.path.exists(img_path):
            generate_gradcam(img_path)
        else:
            print(f"❌ Not found: {img_path}")

Found 1584 test images.


2026-07-05 05:53:58.620880: I external/local_xla/xla/service/service.cc:163] XLA service 0x718d8401c850 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-07-05 05:53:58.620911: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Ti Laptop GPU, Compute Capability 8.6
2026-07-05 05:53:58.790699: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-07-05 05:53:59.892366: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002
2026-07-05 05:54:04.889770: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-05 05:54:05.019629: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kern

1/1 ━━━━━━━━━━━━━━━━━━━━ 19s 19s/step
Predicted: glioma (99.99%)
Using layer: top_conv


/home/abdulhafeez/miniconda3/envs/tf/lib/python3.9/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: [['input_layer']]
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


✅ Grad-CAM saved: gradcam_results/gradcam_glioma1002.png

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Predicted: glioma (99.99%)
Using layer: top_conv
✅ Grad-CAM saved: gradcam_results/gradcam_glioma1008.png

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Predicted: glioma (100.00%)
Using layer: top_conv
✅ Grad-CAM saved: gradcam_results/gradcam_glioma101.png

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Predicted: glioma (100.00%)
Using layer: top_conv
✅ Grad-CAM saved: gradcam_results/gradcam_glioma1027.png

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
Predicted: glioma (100.00%)
Using layer: top_conv
✅ Grad-CAM saved: gradcam_results/gradcam_glioma103.png

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
Predicted: glioma (99.99%)
Using layer: top_conv
✅ Grad-CAM saved: gradcam_results/gradcam_glioma1044.png

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Predicted: glioma (100.00%)
Using layer: top_conv
✅ Grad-CAM saved: gradcam_results/gradcam_glioma1058.png

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Predicted: glioma (99.78%)
Using

In [12]:
# Add this function
def get_true_label_from_path(img_path):
    """Extract true label from folder name"""
    parts = img_path.lower().split(os.sep)
    for cls in CLASS_NAMES:
        if cls in parts:
            return cls
    return "unknown"

# Then update the generate_gradcam function - replace the prediction part with:
    true_label = get_true_label_from_path(img_path)
    preds = model.predict(img_array, verbose=0)
    predicted_class = CLASS_NAMES[np.argmax(preds[0])]
    confidence = np.max(preds[0]) * 100
    
    print(f"True: {true_label} | Predicted: {predicted_class} ({confidence:.2f}%)")